# Notebook 10 — Sensibilidad ante incertidumbre de medición morfométrica

## Grain AI–MDPI

Este notebook evalúa la estabilidad de los modelos congelados ante perturbaciones controladas de las variables morfométricas.

### Modelos principales

- **Dry Bean:** `R1_full_native + rbf_svm`.
- **INIAP Rice:** `R4_hybrid_log_area + rbf_svm`.

### Objetivos

1. Verificar que los modelos guardados reproducen las probabilidades del Notebook 05.
2. Evaluar un **sesgo coherente de escala** de −5%, −2%, −1%, 0%, +1%, +2% y +5%.
3. Evaluar **ruido aleatorio de descriptores** con magnitudes de 1%, 2.5%, 5% y 10% de la dispersión robusta observada en entrenamiento.
4. Perturbar:
   - Todas las variables.
   - Solo el bloque relacionado con tamaño.
   - Solo el bloque relacionado con forma.
5. Medir:
   - Exactitud, macro-F1 y MCC.
   - Cambio de etiquetas respecto de la línea base.
   - Cambio absoluto de probabilidades.
   - Confianza media.
   - Cobertura y riesgo selectivos usando el umbral del 90% fijado en el Notebook 07.
   - Rendimiento por clase.
6. Exportar tablas, figuras, protocolo, validaciones y manifiesto SHA-256.

### Interpretación

El sesgo coherente de escala representa errores sistemáticos de calibración espacial o conversión de unidades. El ruido aleatorio representa incertidumbre en los descriptores ya calculados; no intenta reproducir un proceso específico de segmentación.

Los intervalos de las simulaciones aleatorias describen variabilidad entre perturbaciones. **No son intervalos de confianza muestrales.**

### Restricciones

- No se reentrenan modelos.
- No se reajustan hiperparámetros.
- No se seleccionan niveles de perturbación usando la prueba.
- No se recalibran temperaturas ni umbrales selectivos.
- No se afirma robustez ante cámaras, iluminación o segmentación.
- Las perturbaciones son análisis de sensibilidad en el espacio morfométrico.

In [ ]:
# ============================================================
# 1. Dependencias y entorno
# ============================================================
import importlib
import importlib.metadata
import subprocess
import sys

PINNED_SKLEARN = "1.6.1"

try:
    current_sklearn = importlib.metadata.version("scikit-learn")
except importlib.metadata.PackageNotFoundError:
    current_sklearn = None

packages = []

if current_sklearn != PINNED_SKLEARN:
    packages.append(f"scikit-learn=={PINNED_SKLEARN}")

for module_name, package_name in {
    "pyarrow": "pyarrow",
    "joblib": "joblib",
}.items():
    if importlib.util.find_spec(module_name) is None:
        packages.append(package_name)

if packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import platform
import shutil
import zipfile
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("scikit-learn:", importlib.metadata.version("scikit-learn"))
print("Plataforma:", platform.platform())
print("CPU disponibles:", os.cpu_count())

In [ ]:
# ============================================================
# 2. Configuración reproducible
# ============================================================
RANDOM_SEED = 20260807

N_MONTE_CARLO = 50
PERTURBATION_INTERVAL = 0.95

DATASETS = ["dry_bean", "iniap_rice"]

SCALE_BIAS_LEVELS = [
    -0.05,
    -0.02,
    -0.01,
    0.00,
    0.01,
    0.02,
    0.05,
]

RANDOM_NOISE_LEVELS = [
    0.01,
    0.025,
    0.05,
    0.10,
]

PRIMARY_RANDOM_NOISE_LEVEL = 0.05
PRIMARY_SELECTIVE_TARGET_COVERAGE = 0.90
PRIMARY_PROBABILITY_STATUS = "temperature_scaled"

FEATURE_BLOCKS = {
    "dry_bean": {
        "size_dependent": [
            "Area",
            "Perimeter",
            "MajorAxisLength",
            "MinorAxisLength",
            "ConvexArea",
            "EquivDiameter",
            "ShapeFactor1",
            "ShapeFactor2",
        ],
        "invariant_shape": [
            "AspectRation",
            "Eccentricity",
            "Extent",
            "Solidity",
            "roundness",
            "Compactness",
            "ShapeFactor3",
            "ShapeFactor4",
        ],
    },
    "iniap_rice": {
        "invariant_shape": [
            "aspect_ratio",
            "extent",
            "solidity",
            "circularity",
            "ellipse_fill",
        ],
        "size_anchor": [
            "log_area",
        ],
    },
}

RANDOM_SCENARIOS = {
    dataset_id: {
        "all_features": [
            feature
            for block_features in FEATURE_BLOCKS[dataset_id].values()
            for feature in block_features
        ],
        **FEATURE_BLOCKS[dataset_id],
    }
    for dataset_id in DATASETS
}

DRY_BEAN_SCALE_EXPONENTS = {
    "Area": 2.0,
    "Perimeter": 1.0,
    "MajorAxisLength": 1.0,
    "MinorAxisLength": 1.0,
    "AspectRation": 0.0,
    "Eccentricity": 0.0,
    "ConvexArea": 2.0,
    "EquivDiameter": 1.0,
    "Extent": 0.0,
    "Solidity": 0.0,
    "roundness": 0.0,
    "Compactness": 0.0,
    "ShapeFactor1": -1.0,
    "ShapeFactor2": -1.0,
    "ShapeFactor3": 0.0,
    "ShapeFactor4": 0.0,
}

UNIT_INTERVAL_FEATURES = {
    "dry_bean": {
        "Eccentricity",
        "Extent",
        "Solidity",
        "roundness",
        "Compactness",
        "ShapeFactor3",
        "ShapeFactor4",
    },
    "iniap_rice": {
        "extent",
        "solidity",
        "circularity",
        "ellipse_fill",
    },
}

POSITIVE_FEATURES = {
    "dry_bean": set(FEATURE_BLOCKS["dry_bean"]["size_dependent"])
    | {"AspectRation"},
    "iniap_rice": {"aspect_ratio"},
}

LOCAL_ROOT = Path("/content/Grain_AI_MDPI_notebook10")
INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT03_DIR = LOCAL_ROOT / "01_extracted_notebook03"
EXTRACT04_DIR = LOCAL_ROOT / "02_extracted_notebook04"
EXTRACT05_DIR = LOCAL_ROOT / "03_extracted_notebook05"
EXTRACT07_DIR = LOCAL_ROOT / "04_extracted_notebook07"
EXTRACT09_DIR = LOCAL_ROOT / "05_extracted_notebook09"
OUTPUT_ROOT = LOCAL_ROOT / "06_outputs"

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "10_MEASUREMENT_UNCERTAINTY"
FIGURES_DIR = OUTPUT_ROOT / "08_figures" / "10_MEASUREMENT_UNCERTAINTY"
TABLES_DIR = OUTPUT_ROOT / "09_tables" / "10_MEASUREMENT_UNCERTAINTY"

for directory in [
    INPUT_DIR,
    EXTRACT03_DIR,
    EXTRACT04_DIR,
    EXTRACT05_DIR,
    EXTRACT07_DIR,
    EXTRACT09_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Réplicas Monte Carlo:", N_MONTE_CARLO)
print("Sesgos de escala:", SCALE_BIAS_LEVELS)
print("Niveles de ruido:", RANDOM_NOISE_LEVELS)

In [ ]:
# ============================================================
# 3. Google Drive y localización de entradas
# ============================================================
try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None
    print("No se detectó Google Colab.")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Grain_Robustness_Q1")
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

ZIP_FILENAMES = {
    "notebook03": "NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip",
    "notebook04": "NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip",
    "notebook05": "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip",
    "notebook07": "NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS.zip",
    "notebook09": "NOTEBOOK09_ABLATION_INTERPRETABILITY_OUTPUTS.zip",
}

def find_exact_file(root: Path, filename: str):
    if not root.exists():
        return None

    matches = sorted(root.rglob(filename))

    if not matches:
        return None

    return max(
        matches,
        key=lambda path: path.stat().st_mtime,
    )

def obtain_input_zip(filename: str) -> Path:
    found = find_exact_file(
        DRIVE_PROJECT_ROOT,
        filename,
    )

    if found is not None:
        print("Encontrado en Drive:", found)
        return found

    local_candidate = Path("/content") / filename

    if local_candidate.exists():
        print("Encontrado en /content:", local_candidate)
        return local_candidate

    if IN_COLAB and files is not None:
        print(
            f"No se encontró {filename}. "
            "Selecciónelo para subirlo."
        )

        uploaded = files.upload()

        if filename not in uploaded:
            raise FileNotFoundError(
                f"Debe subir exactamente {filename}."
            )

        destination = INPUT_DIR / filename
        destination.write_bytes(uploaded[filename])
        return destination

    raise FileNotFoundError(
        f"No se encontró {filename}."
    )

zip_paths = {
    key: obtain_input_zip(filename)
    for key, filename in ZIP_FILENAMES.items()
}

display(
    pd.DataFrame(
        [
            {
                "source": key,
                "path": str(path),
            }
            for key, path in zip_paths.items()
        ]
    )
)

In [ ]:
# ============================================================
# 4. Extracción y validaciones previas
# ============================================================
def clean_extract(zip_path: Path, destination: Path):
    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)

extract_map = {
    "notebook03": EXTRACT03_DIR,
    "notebook04": EXTRACT04_DIR,
    "notebook05": EXTRACT05_DIR,
    "notebook07": EXTRACT07_DIR,
    "notebook09": EXTRACT09_DIR,
}

for key, destination in extract_map.items():
    clean_extract(
        zip_paths[key],
        destination,
    )

SELECTED_CONFIG_FILE = (
    EXTRACT04_DIR
    / "07_results"
    / "04_MODEL_SELECTION"
    / "03_selected_model_configurations.csv"
)

NOTEBOOK07_VALIDATION_FILE = (
    EXTRACT07_DIR
    / "07_results"
    / "07_SELECTIVE_UNCERTAINTY"
    / "05_final_selective_validation.csv"
)

SELECTED_SCORE_FILE = (
    EXTRACT07_DIR
    / "07_results"
    / "07_SELECTIVE_UNCERTAINTY"
    / "02_selected_certainty_scores.csv"
)

OPERATING_POINT_FILE = (
    EXTRACT07_DIR
    / "09_tables"
    / "07_SELECTIVE_UNCERTAINTY"
    / "Table_calibration_fixed_operating_points.csv"
)

NOTEBOOK09_VALIDATION_FILE = (
    EXTRACT09_DIR
    / "07_results"
    / "09_ABLATION_INTERPRETABILITY"
    / "06_final_interpretability_validation.csv"
)

required_files = [
    SELECTED_CONFIG_FILE,
    NOTEBOOK07_VALIDATION_FILE,
    SELECTED_SCORE_FILE,
    OPERATING_POINT_FILE,
    NOTEBOOK09_VALIDATION_FILE,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Faltan archivos requeridos:\n"
        + "\n".join(missing_files)
    )

selected_configurations = pd.read_csv(
    SELECTED_CONFIG_FILE
)

notebook07_validation = pd.read_csv(
    NOTEBOOK07_VALIDATION_FILE
)

selected_certainty_scores = pd.read_csv(
    SELECTED_SCORE_FILE
)

selective_operating_points = pd.read_csv(
    OPERATING_POINT_FILE
)

notebook09_validation = pd.read_csv(
    NOTEBOOK09_VALIDATION_FILE
)

if not notebook07_validation["passed"].astype(bool).all():
    display(notebook07_validation)
    raise RuntimeError(
        "El Notebook 07 no superó sus validaciones."
    )

if not notebook09_validation["passed"].astype(bool).all():
    display(notebook09_validation)
    raise RuntimeError(
        "El Notebook 09 no superó sus validaciones."
    )

print("Notebooks 07 y 09 validados.")

In [ ]:
# ============================================================
# 5. Carga de campeones, modelos y unidades de grupo
# ============================================================
training_champions = (
    selected_configurations.sort_values(
        [
            "dataset_id",
            "macro_f1_mean",
            "log_loss_mean",
            "candidate_id",
        ],
        ascending=[
            True,
            False,
            True,
            True,
        ],
    )
    .groupby(
        "dataset_id",
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

if len(training_champions) != len(DATASETS):
    display(training_champions)
    raise RuntimeError(
        "Número inesperado de campeones."
    )

def load_json(path: Path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )

bundles = {}

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    split_path = (
        EXTRACT03_DIR
        / "05_frozen_splits"
        / dataset_id
        / f"{representation}_with_split.parquet"
    )

    model_path = (
        EXTRACT05_DIR
        / "05_models"
        / dataset_id
        / representation
        / f"{algorithm}.joblib"
    )

    metadata_path = (
        EXTRACT05_DIR
        / "05_models"
        / dataset_id
        / representation
        / f"{algorithm}_metadata.json"
    )

    prediction_path = (
        EXTRACT05_DIR
        / "06_predictions"
        / "05_BASELINE"
        / dataset_id
        / representation
        / f"{algorithm}_test_predictions.parquet"
    )

    for path in [
        split_path,
        model_path,
        metadata_path,
        prediction_path,
    ]:
        if not path.exists():
            raise FileNotFoundError(path)

    frame = pd.read_parquet(split_path)
    model = joblib.load(model_path)
    metadata = load_json(metadata_path)
    saved_test_predictions = pd.read_parquet(
        prediction_path
    )

    feature_columns = [
        str(feature)
        for feature in metadata["feature_columns"]
    ]

    class_order = [
        str(label)
        for label in metadata["class_order"]
    ]

    expected_features = set(
        feature
        for features in FEATURE_BLOCKS[dataset_id].values()
        for feature in features
    )

    if expected_features != set(feature_columns):
        raise RuntimeError(
            f"Los bloques no cubren las variables de {dataset_id}."
        )

    group_frames = {}

    for partition in [
        "train",
        "calibration",
        "test",
    ]:
        partition_frame = (
            frame.loc[
                frame["partition"] == partition
            ]
            .sort_values("record_id")
            .reset_index(drop=True)
        )

        label_counts = (
            partition_frame.groupby(
                "duplicate_group_id"
            )["class_label"].nunique()
        )

        vector_counts = (
            partition_frame.groupby(
                "duplicate_group_id"
            )[feature_columns]
            .nunique(dropna=False)
            .max(axis=1)
        )

        if (label_counts > 1).any():
            raise RuntimeError(
                f"Grupo multiclase: {dataset_id}, {partition}."
            )

        if (vector_counts > 1).any():
            raise RuntimeError(
                f"Grupo con vectores distintos: {dataset_id}, {partition}."
            )

        group_frames[partition] = (
            partition_frame.groupby(
                "duplicate_group_id",
                as_index=False,
                sort=True,
            )
            .first()
        )

    bundles[dataset_id] = {
        "representation": representation,
        "algorithm": algorithm,
        "candidate_id": champion["candidate_id"],
        "model": model,
        "metadata": metadata,
        "feature_columns": feature_columns,
        "class_order": class_order,
        "temperature": float(metadata["temperature"]),
        "group_frames": group_frames,
        "saved_test_predictions": saved_test_predictions,
    }

display(
    training_champions[
        [
            "dataset_id",
            "representation",
            "algorithm",
            "candidate_id",
            "macro_f1_mean",
            "macro_f1_sd",
        ]
    ]
)

In [ ]:
# ============================================================
# 6. Funciones de probabilidad, métricas y auditoría
# ============================================================
def aligned_model_probabilities(
    model,
    X,
    class_order,
):
    X = np.asarray(X, dtype=float)

    raw_probabilities = np.asarray(
        model.predict_proba(X),
        dtype=float,
    )

    model_classes = np.asarray(
        model.classes_,
        dtype=str,
    )

    destination = {
        str(class_label): class_index
        for class_index, class_label
        in enumerate(class_order)
    }

    if set(model_classes) != set(class_order):
        raise RuntimeError(
            "Las clases del modelo no coinciden con el metadata."
        )

    aligned = np.zeros(
        (
            len(X),
            len(class_order),
        ),
        dtype=float,
    )

    for source_index, class_label in enumerate(
        model_classes
    ):
        aligned[
            :,
            destination[str(class_label)],
        ] = raw_probabilities[
            :,
            source_index,
        ]

    aligned = aligned / aligned.sum(
        axis=1,
        keepdims=True,
    )

    return aligned

def probability_temperature_scale(
    probabilities,
    temperature,
):
    clipped = np.clip(
        probabilities,
        1e-15,
        1.0,
    )

    return softmax(
        np.log(clipped)
        / float(temperature),
        axis=1,
    )

def labels_from_probabilities(
    probabilities,
    class_order,
):
    return np.asarray(
        class_order,
        dtype=str,
    )[
        np.asarray(
            probabilities
        ).argmax(axis=1)
    ]

def certainty_score(
    probabilities,
    score_name,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    sorted_probabilities = np.sort(
        probabilities,
        axis=1,
    )

    if score_name == "max_probability":
        return sorted_probabilities[:, -1]

    if score_name == "top2_margin":
        return (
            sorted_probabilities[:, -1]
            - sorted_probabilities[:, -2]
        )

    if score_name == "one_minus_normalized_entropy":
        clipped = np.clip(
            probabilities,
            1e-15,
            1.0,
        )

        entropy = -np.sum(
            clipped * np.log(clipped),
            axis=1,
        )

        return (
            1.0
            - entropy
            / np.log(
                probabilities.shape[1]
            )
        )

    raise ValueError(score_name)

def macro_f1(
    y_true,
    y_pred,
    labels,
):
    return float(
        f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        )
    )

def evaluate_probabilities(
    y_true,
    native_probabilities,
    calibrated_probabilities,
    class_order,
    baseline_labels,
    baseline_native_probabilities,
    selected_score_name,
    selective_threshold,
):
    y_true = np.asarray(
        y_true,
        dtype=str,
    )

    prediction_labels = (
        labels_from_probabilities(
            native_probabilities,
            class_order,
        )
    )

    accuracy = float(
        accuracy_score(
            y_true,
            prediction_labels,
        )
    )

    score_values = certainty_score(
        calibrated_probabilities,
        selected_score_name,
    )

    retained = (
        score_values
        >= selective_threshold
    )

    n_retained = int(
        retained.sum()
    )

    if n_retained > 0:
        selective_accuracy = float(
            accuracy_score(
                y_true[retained],
                prediction_labels[retained],
            )
        )

        selective_macro_f1 = macro_f1(
            y_true[retained],
            prediction_labels[retained],
            class_order,
        )
    else:
        selective_accuracy = np.nan
        selective_macro_f1 = np.nan

    class_recalls = []

    for class_label in class_order:
        class_mask = (
            y_true == class_label
        )

        if class_mask.any():
            class_recalls.append(
                float(
                    np.mean(
                        prediction_labels[class_mask]
                        == class_label
                    )
                )
            )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1(
            y_true,
            prediction_labels,
            class_order,
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                prediction_labels,
            )
        ),
        "balanced_accuracy": float(
            np.mean(class_recalls)
        ),
        "label_flip_rate": float(
            np.mean(
                prediction_labels
                != baseline_labels
            )
        ),
        "mean_absolute_probability_shift": float(
            np.mean(
                np.abs(
                    native_probabilities
                    - baseline_native_probabilities
                )
            )
        ),
        "mean_confidence": float(
            calibrated_probabilities.max(
                axis=1
            ).mean()
        ),
        "selective_coverage": float(
            retained.mean()
        ),
        "selective_accuracy": (
            selective_accuracy
        ),
        "selective_risk": (
            1.0 - selective_accuracy
            if np.isfinite(
                selective_accuracy
            )
            else np.nan
        ),
        "selective_macro_f1": (
            selective_macro_f1
        ),
        "n_retained": n_retained,
    }

def classwise_metrics(
    y_true,
    prediction_labels,
    class_order,
):
    rows = []

    for class_label in class_order:
        mask = (
            y_true == class_label
        )

        support = int(mask.sum())

        recall = float(
            np.mean(
                prediction_labels[mask]
                == class_label
            )
        )

        rows.append(
            {
                "class_label": class_label,
                "support": support,
                "recall": recall,
                "error_rate": 1.0 - recall,
            }
        )

    return pd.DataFrame(rows)

# ------------------------------------------------------------
# Auditoría de fidelidad y configuración selectiva
# ------------------------------------------------------------
fidelity_rows = []
selective_configuration_rows = []

for dataset_id, bundle in bundles.items():
    test_frame = bundle[
        "group_frames"
    ]["test"]

    X_test = test_frame[
        bundle["feature_columns"]
    ].to_numpy(dtype=float)

    native_probabilities = (
        aligned_model_probabilities(
            bundle["model"],
            X_test,
            bundle["class_order"],
        )
    )

    saved = (
        bundle[
            "saved_test_predictions"
        ]
        .copy()
    )

    saved[
        "duplicate_group_id"
    ] = saved[
        "duplicate_group_id"
    ].astype(str)

    saved_group = (
        saved.sort_values("record_id")
        .groupby(
            "duplicate_group_id",
            as_index=False,
            sort=True,
        )
        .first()
    )

    probability_columns = [
        f"prob_native_{index}"
        for index in range(
            len(
                bundle[
                    "class_order"
                ]
            )
        )
    ]

    saved_probabilities = saved_group[
        probability_columns
    ].to_numpy(dtype=float)

    max_difference = float(
        np.max(
            np.abs(
                native_probabilities
                - saved_probabilities
            )
        )
    )

    fidelity_rows.append(
        {
            "dataset_id": dataset_id,
            "test_groups": len(test_frame),
            "max_probability_difference": max_difference,
            "passed": bool(
                max_difference <= 1e-10
            ),
        }
    )

    selected_score_row = (
        selected_certainty_scores.loc[
            (
                selected_certainty_scores[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                selected_certainty_scores[
                    "probability_status"
                ] == PRIMARY_PROBABILITY_STATUS
            )
        ]
    )

    if len(selected_score_row) != 1:
        raise RuntimeError(
            f"Score selectivo ambiguo para {dataset_id}."
        )

    score_name = selected_score_row[
        "certainty_score"
    ].iloc[0]

    threshold_row = (
        selective_operating_points.loc[
            (
                selective_operating_points[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                selective_operating_points[
                    "probability_status"
                ] == PRIMARY_PROBABILITY_STATUS
            )
            & (
                selective_operating_points[
                    "certainty_score"
                ] == score_name
            )
            & (
                selective_operating_points[
                    "selected_in_calibration"
                ].astype(bool)
            )
            & np.isclose(
                selective_operating_points[
                    "target_calibration_coverage"
                ],
                PRIMARY_SELECTIVE_TARGET_COVERAGE,
            )
        ]
    )

    if len(threshold_row) != 1:
        raise RuntimeError(
            f"Umbral selectivo ambiguo para {dataset_id}."
        )

    selective_configuration_rows.append(
        {
            "dataset_id": dataset_id,
            "certainty_score": score_name,
            "target_calibration_coverage": (
                PRIMARY_SELECTIVE_TARGET_COVERAGE
            ),
            "certainty_threshold": float(
                threshold_row[
                    "certainty_threshold"
                ].iloc[0]
            ),
            "temperature": bundle[
                "temperature"
            ],
        }
    )

model_fidelity = pd.DataFrame(
    fidelity_rows
)

selective_configuration = pd.DataFrame(
    selective_configuration_rows
)

if not model_fidelity[
    "passed"
].astype(bool).all():
    display(model_fidelity)
    raise RuntimeError(
        "Los modelos no reproducen el Notebook 05."
    )

model_fidelity.to_csv(
    RESULTS_DIR
    / "01_model_fidelity_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

selective_configuration.to_csv(
    RESULTS_DIR
    / "02_fixed_selective_configuration.csv",
    index=False,
    encoding="utf-8-sig",
)

display(model_fidelity)
display(selective_configuration)

In [ ]:
# ============================================================
# 7. Línea base y escalas robustas de entrenamiento
# ============================================================
baseline_rows = []
baseline_class_rows = []
baseline_cache = {}
robust_scale_rows = []

for dataset_id, bundle in bundles.items():
    feature_columns = bundle[
        "feature_columns"
    ]

    class_order = bundle[
        "class_order"
    ]

    train_frame = bundle[
        "group_frames"
    ]["train"]

    test_frame = bundle[
        "group_frames"
    ]["test"]

    X_train = train_frame[
        feature_columns
    ].to_numpy(dtype=float)

    X_test = test_frame[
        feature_columns
    ].to_numpy(dtype=float)

    y_test = test_frame[
        "class_label"
    ].astype(str).to_numpy()

    native_probabilities = (
        aligned_model_probabilities(
            bundle["model"],
            X_test,
            class_order,
        )
    )

    calibrated_probabilities = (
        probability_temperature_scale(
            native_probabilities,
            bundle["temperature"],
        )
    )

    baseline_labels = (
        labels_from_probabilities(
            native_probabilities,
            class_order,
        )
    )

    selective_row = (
        selective_configuration.loc[
            selective_configuration[
                "dataset_id"
            ] == dataset_id
        ].iloc[0]
    )

    baseline_metrics = evaluate_probabilities(
        y_true=y_test,
        native_probabilities=(
            native_probabilities
        ),
        calibrated_probabilities=(
            calibrated_probabilities
        ),
        class_order=class_order,
        baseline_labels=baseline_labels,
        baseline_native_probabilities=(
            native_probabilities
        ),
        selected_score_name=(
            selective_row[
                "certainty_score"
            ]
        ),
        selective_threshold=float(
            selective_row[
                "certainty_threshold"
            ]
        ),
    )

    baseline_rows.append(
        {
            "dataset_id": dataset_id,
            "representation": bundle[
                "representation"
            ],
            "algorithm": bundle[
                "algorithm"
            ],
            "test_groups": len(
                test_frame
            ),
            **baseline_metrics,
        }
    )

    class_table = classwise_metrics(
        y_true=y_test,
        prediction_labels=baseline_labels,
        class_order=class_order,
    )

    class_table.insert(
        0,
        "condition",
        "baseline",
    )

    class_table.insert(
        0,
        "dataset_id",
        dataset_id,
    )

    baseline_class_rows.append(
        class_table
    )

    q25 = np.quantile(
        X_train,
        0.25,
        axis=0,
    )

    q75 = np.quantile(
        X_train,
        0.75,
        axis=0,
    )

    robust_scale = (
        q75 - q25
    ) / 1.349

    standard_scale = np.std(
        X_train,
        axis=0,
        ddof=1,
    )

    robust_scale = np.where(
        robust_scale > 0,
        robust_scale,
        standard_scale,
    )

    robust_scale = np.where(
        robust_scale > 0,
        robust_scale,
        1.0,
    )

    robust_scale_by_feature = {
        feature: float(scale)
        for feature, scale
        in zip(
            feature_columns,
            robust_scale,
        )
    }

    for feature in feature_columns:
        robust_scale_rows.append(
            {
                "dataset_id": dataset_id,
                "feature": feature,
                "training_robust_scale": (
                    robust_scale_by_feature[
                        feature
                    ]
                ),
            }
        )

    baseline_cache[
        dataset_id
    ] = {
        "X_test": X_test,
        "y_test": y_test,
        "baseline_native_probabilities": (
            native_probabilities
        ),
        "baseline_calibrated_probabilities": (
            calibrated_probabilities
        ),
        "baseline_labels": baseline_labels,
        "robust_scale_by_feature": (
            robust_scale_by_feature
        ),
        "feature_to_index": {
            feature: index
            for index, feature
            in enumerate(
                feature_columns
            )
        },
        "selective_score_name": (
            selective_row[
                "certainty_score"
            ]
        ),
        "selective_threshold": float(
            selective_row[
                "certainty_threshold"
            ]
        ),
    }

baseline_results = pd.DataFrame(
    baseline_rows
)

baseline_class_results = pd.concat(
    baseline_class_rows,
    ignore_index=True,
)

training_robust_scales = pd.DataFrame(
    robust_scale_rows
)

baseline_results.to_csv(
    TABLES_DIR
    / "Table_baseline_group_level_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

baseline_class_results.to_csv(
    TABLES_DIR
    / "Table_baseline_classwise_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

training_robust_scales.to_csv(
    RESULTS_DIR
    / "03_training_robust_feature_scales.csv",
    index=False,
    encoding="utf-8-sig",
)

display(baseline_results)

In [ ]:
# ============================================================
# 8. Funciones de perturbación
# ============================================================
def apply_physical_bounds(
    X,
    dataset_id,
    feature_columns,
):
    X = np.asarray(
        X,
        dtype=float,
    ).copy()

    feature_to_index = {
        feature: index
        for index, feature
        in enumerate(feature_columns)
    }

    for feature in UNIT_INTERVAL_FEATURES[
        dataset_id
    ]:
        index = feature_to_index[
            feature
        ]

        X[:, index] = np.clip(
            X[:, index],
            0.0,
            1.0,
        )

    for feature in POSITIVE_FEATURES[
        dataset_id
    ]:
        index = feature_to_index[
            feature
        ]

        X[:, index] = np.maximum(
            X[:, index],
            np.finfo(float).tiny,
        )

    return X

def apply_coherent_scale_bias(
    X,
    dataset_id,
    feature_columns,
    relative_bias,
):
    if relative_bias <= -1.0:
        raise ValueError(
            "El sesgo relativo debe ser mayor que -1."
        )

    X_perturbed = np.asarray(
        X,
        dtype=float,
    ).copy()

    feature_to_index = {
        feature: index
        for index, feature
        in enumerate(feature_columns)
    }

    scale_factor = (
        1.0
        + float(relative_bias)
    )

    if dataset_id == "dry_bean":
        for feature, exponent in (
            DRY_BEAN_SCALE_EXPONENTS.items()
        ):
            index = feature_to_index[
                feature
            ]

            X_perturbed[
                :,
                index,
            ] = (
                X_perturbed[
                    :,
                    index,
                ]
                * (
                    scale_factor
                    ** exponent
                )
            )

    elif dataset_id == "iniap_rice":
        log_area_index = (
            feature_to_index[
                "log_area"
            ]
        )

        X_perturbed[
            :,
            log_area_index,
        ] = (
            X_perturbed[
                :,
                log_area_index,
            ]
            + 2.0
            * np.log(
                scale_factor
            )
        )

    else:
        raise ValueError(
            dataset_id
        )

    return apply_physical_bounds(
        X_perturbed,
        dataset_id,
        feature_columns,
    )

def apply_random_descriptor_noise(
    X,
    dataset_id,
    feature_columns,
    selected_features,
    robust_scale_by_feature,
    noise_level,
    rng,
):
    X_perturbed = np.asarray(
        X,
        dtype=float,
    ).copy()

    feature_to_index = {
        feature: index
        for index, feature
        in enumerate(feature_columns)
    }

    for feature in selected_features:
        index = feature_to_index[
            feature
        ]

        standard_deviation = (
            float(noise_level)
            * robust_scale_by_feature[
                feature
            ]
        )

        X_perturbed[
            :,
            index,
        ] = (
            X_perturbed[
                :,
                index,
            ]
            + rng.normal(
                loc=0.0,
                scale=standard_deviation,
                size=len(X_perturbed),
            )
        )

    return apply_physical_bounds(
        X_perturbed,
        dataset_id,
        feature_columns,
    )

def evaluate_perturbed_matrix(
    dataset_id,
    X_perturbed,
):
    bundle = bundles[
        dataset_id
    ]

    cache = baseline_cache[
        dataset_id
    ]

    native_probabilities = (
        aligned_model_probabilities(
            bundle["model"],
            X_perturbed,
            bundle["class_order"],
        )
    )

    calibrated_probabilities = (
        probability_temperature_scale(
            native_probabilities,
            bundle["temperature"],
        )
    )

    return evaluate_probabilities(
        y_true=cache[
            "y_test"
        ],
        native_probabilities=(
            native_probabilities
        ),
        calibrated_probabilities=(
            calibrated_probabilities
        ),
        class_order=bundle[
            "class_order"
        ],
        baseline_labels=cache[
            "baseline_labels"
        ],
        baseline_native_probabilities=(
            cache[
                "baseline_native_probabilities"
            ]
        ),
        selected_score_name=cache[
            "selective_score_name"
        ],
        selective_threshold=cache[
            "selective_threshold"
        ],
    )

In [ ]:
# ============================================================
# 9. Sensibilidad a sesgo coherente de escala
# ============================================================
scale_bias_rows = []
scale_bias_class_rows = []

for dataset_id, bundle in bundles.items():
    cache = baseline_cache[
        dataset_id
    ]

    for relative_bias in SCALE_BIAS_LEVELS:
        X_perturbed = (
            apply_coherent_scale_bias(
                X=cache[
                    "X_test"
                ],
                dataset_id=dataset_id,
                feature_columns=bundle[
                    "feature_columns"
                ],
                relative_bias=relative_bias,
            )
        )

        metrics = evaluate_perturbed_matrix(
            dataset_id,
            X_perturbed,
        )

        baseline_macro_f1 = float(
            baseline_results.loc[
                baseline_results[
                    "dataset_id"
                ] == dataset_id,
                "macro_f1",
            ].iloc[0]
        )

        baseline_selective_risk = float(
            baseline_results.loc[
                baseline_results[
                    "dataset_id"
                ] == dataset_id,
                "selective_risk",
            ].iloc[0]
        )

        scale_bias_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": bundle[
                    "representation"
                ],
                "algorithm": bundle[
                    "algorithm"
                ],
                "relative_scale_bias": (
                    relative_bias
                ),
                "scale_bias_percent": (
                    100.0
                    * relative_bias
                ),
                "macro_f1_drop": (
                    baseline_macro_f1
                    - metrics[
                        "macro_f1"
                    ]
                ),
                "selective_risk_increase": (
                    metrics[
                        "selective_risk"
                    ]
                    - baseline_selective_risk
                ),
                **metrics,
            }
        )

        native_probabilities = (
            aligned_model_probabilities(
                bundle["model"],
                X_perturbed,
                bundle[
                    "class_order"
                ],
            )
        )

        prediction_labels = (
            labels_from_probabilities(
                native_probabilities,
                bundle[
                    "class_order"
                ],
            )
        )

        class_table = (
            classwise_metrics(
                y_true=cache[
                    "y_test"
                ],
                prediction_labels=(
                    prediction_labels
                ),
                class_order=bundle[
                    "class_order"
                ],
            )
        )

        class_table.insert(
            0,
            "scale_bias_percent",
            100.0 * relative_bias,
        )

        class_table.insert(
            0,
            "dataset_id",
            dataset_id,
        )

        scale_bias_class_rows.append(
            class_table
        )

scale_bias_results = pd.DataFrame(
    scale_bias_rows
)

scale_bias_class_results = pd.concat(
    scale_bias_class_rows,
    ignore_index=True,
)

scale_bias_results.to_csv(
    TABLES_DIR
    / "Table_coherent_scale_bias_sensitivity.csv",
    index=False,
    encoding="utf-8-sig",
)

scale_bias_class_results.to_csv(
    TABLES_DIR
    / "Table_coherent_scale_bias_classwise.csv",
    index=False,
    encoding="utf-8-sig",
)

display(scale_bias_results)

In [ ]:
# ============================================================
# 10. Ruido aleatorio de descriptores — Monte Carlo
# ============================================================
random_noise_rows = []
random_class_rows = []

total_runs = sum(
    len(RANDOM_SCENARIOS[dataset_id])
    * len(RANDOM_NOISE_LEVELS)
    * N_MONTE_CARLO
    for dataset_id in DATASETS
)

completed_runs = 0

for dataset_index, dataset_id in enumerate(
    DATASETS
):
    bundle = bundles[
        dataset_id
    ]

    cache = baseline_cache[
        dataset_id
    ]

    baseline_macro_f1 = float(
        baseline_results.loc[
            baseline_results[
                "dataset_id"
            ] == dataset_id,
            "macro_f1",
        ].iloc[0]
    )

    baseline_selective_risk = float(
        baseline_results.loc[
            baseline_results[
                "dataset_id"
            ] == dataset_id,
            "selective_risk",
        ].iloc[0]
    )

    for scenario_index, (
        scenario_name,
        selected_features,
    ) in enumerate(
        RANDOM_SCENARIOS[
            dataset_id
        ].items()
    ):
        for level_index, noise_level in enumerate(
            RANDOM_NOISE_LEVELS
        ):
            for replicate in range(
                N_MONTE_CARLO
            ):
                seed = (
                    RANDOM_SEED
                    + dataset_index
                    * 10_000_000
                    + scenario_index
                    * 1_000_000
                    + level_index
                    * 10_000
                    + replicate
                )

                rng = np.random.default_rng(
                    seed
                )

                X_perturbed = (
                    apply_random_descriptor_noise(
                        X=cache[
                            "X_test"
                        ],
                        dataset_id=dataset_id,
                        feature_columns=bundle[
                            "feature_columns"
                        ],
                        selected_features=(
                            selected_features
                        ),
                        robust_scale_by_feature=(
                            cache[
                                "robust_scale_by_feature"
                            ]
                        ),
                        noise_level=noise_level,
                        rng=rng,
                    )
                )

                metrics = (
                    evaluate_perturbed_matrix(
                        dataset_id,
                        X_perturbed,
                    )
                )

                random_noise_rows.append(
                    {
                        "dataset_id": dataset_id,
                        "representation": bundle[
                            "representation"
                        ],
                        "algorithm": bundle[
                            "algorithm"
                        ],
                        "scenario": scenario_name,
                        "features": " | ".join(
                            selected_features
                        ),
                        "noise_level_fraction_robust_scale": (
                            noise_level
                        ),
                        "noise_level_percent_robust_scale": (
                            100.0
                            * noise_level
                        ),
                        "replicate": replicate,
                        "seed": seed,
                        "macro_f1_drop": (
                            baseline_macro_f1
                            - metrics[
                                "macro_f1"
                            ]
                        ),
                        "selective_risk_increase": (
                            metrics[
                                "selective_risk"
                            ]
                            - baseline_selective_risk
                        ),
                        **metrics,
                    }
                )

                if np.isclose(
                    noise_level,
                    PRIMARY_RANDOM_NOISE_LEVEL,
                ):
                    native_probabilities = (
                        aligned_model_probabilities(
                            bundle[
                                "model"
                            ],
                            X_perturbed,
                            bundle[
                                "class_order"
                            ],
                        )
                    )

                    prediction_labels = (
                        labels_from_probabilities(
                            native_probabilities,
                            bundle[
                                "class_order"
                            ],
                        )
                    )

                    class_table = (
                        classwise_metrics(
                            y_true=cache[
                                "y_test"
                            ],
                            prediction_labels=(
                                prediction_labels
                            ),
                            class_order=bundle[
                                "class_order"
                            ],
                        )
                    )

                    class_table.insert(
                        0,
                        "replicate",
                        replicate,
                    )

                    class_table.insert(
                        0,
                        "scenario",
                        scenario_name,
                    )

                    class_table.insert(
                        0,
                        "noise_level_fraction_robust_scale",
                        noise_level,
                    )

                    class_table.insert(
                        0,
                        "dataset_id",
                        dataset_id,
                    )

                    random_class_rows.append(
                        class_table
                    )

                completed_runs += 1

            print(
                f"{dataset_id} | {scenario_name} | "
                f"{100 * noise_level:.1f}% completado "
                f"({completed_runs}/{total_runs})"
            )

random_noise_raw = pd.DataFrame(
    random_noise_rows
)

random_class_raw = pd.concat(
    random_class_rows,
    ignore_index=True,
)

random_noise_raw.to_csv(
    RESULTS_DIR
    / "04_random_descriptor_noise_all_replicates.csv",
    index=False,
    encoding="utf-8-sig",
)

random_class_raw.to_csv(
    RESULTS_DIR
    / "05_random_noise_classwise_primary_level_replicates.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Simulaciones completadas:",
    len(random_noise_raw),
)

In [ ]:
# ============================================================
# 11. Resumen de perturbaciones aleatorias
# ============================================================
SUMMARY_METRICS = [
    "accuracy",
    "macro_f1",
    "mcc",
    "balanced_accuracy",
    "macro_f1_drop",
    "label_flip_rate",
    "mean_absolute_probability_shift",
    "mean_confidence",
    "selective_coverage",
    "selective_accuracy",
    "selective_risk",
    "selective_risk_increase",
    "selective_macro_f1",
]

def summarize_values(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    alpha = (
        1.0
        - PERTURBATION_INTERVAL
    ) / 2.0

    return {
        "mean": float(
            values.mean()
        ),
        "sd": float(
            values.std(ddof=1)
        ),
        "interval_low": float(
            np.quantile(
                values,
                alpha,
            )
        ),
        "interval_high": float(
            np.quantile(
                values,
                1.0 - alpha,
            )
        ),
        "minimum": float(
            values.min()
        ),
        "maximum": float(
            values.max()
        ),
        "n_replicates": int(
            len(values)
        ),
    }

summary_rows = []

grouping_columns = [
    "dataset_id",
    "representation",
    "algorithm",
    "scenario",
    "features",
    "noise_level_fraction_robust_scale",
    "noise_level_percent_robust_scale",
]

for group_key, subset in (
    random_noise_raw.groupby(
        grouping_columns,
        sort=True,
    )
):
    group_metadata = dict(
        zip(
            grouping_columns,
            group_key,
        )
    )

    for metric in SUMMARY_METRICS:
        summary_rows.append(
            {
                **group_metadata,
                "metric": metric,
                **summarize_values(
                    subset[
                        metric
                    ].to_numpy(dtype=float)
                ),
            }
        )

random_noise_summary = pd.DataFrame(
    summary_rows
)

class_summary_rows = []

class_group_columns = [
    "dataset_id",
    "scenario",
    "noise_level_fraction_robust_scale",
    "class_label",
]

for group_key, subset in (
    random_class_raw.groupby(
        class_group_columns,
        sort=True,
    )
):
    metadata = dict(
        zip(
            class_group_columns,
            group_key,
        )
    )

    for metric in [
        "recall",
        "error_rate",
    ]:
        class_summary_rows.append(
            {
                **metadata,
                "metric": metric,
                **summarize_values(
                    subset[
                        metric
                    ].to_numpy(dtype=float)
                ),
            }
        )

random_class_summary = pd.DataFrame(
    class_summary_rows
)

random_noise_summary.to_csv(
    TABLES_DIR
    / "Table_random_descriptor_noise_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

random_class_summary.to_csv(
    TABLES_DIR
    / "Table_random_noise_classwise_primary_level.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    random_noise_summary.loc[
        random_noise_summary[
            "metric"
        ].isin(
            [
                "macro_f1",
                "macro_f1_drop",
                "selective_coverage",
                "selective_risk",
            ]
        )
    ].head(40)
)

In [ ]:
# ============================================================
# 12. Tablas primarias y puntos de degradación preespecificados
# ============================================================
primary_random_results = (
    random_noise_summary.loc[
        (
            random_noise_summary[
                "scenario"
            ] == "all_features"
        )
        & (
            random_noise_summary[
                "metric"
            ].isin(
                [
                    "macro_f1",
                    "macro_f1_drop",
                    "label_flip_rate",
                    "selective_coverage",
                    "selective_risk",
                    "selective_risk_increase",
                ]
            )
        )
    ]
    .copy()
)

primary_scale_results = (
    scale_bias_results[
        [
            "dataset_id",
            "scale_bias_percent",
            "macro_f1",
            "macro_f1_drop",
            "label_flip_rate",
            "selective_coverage",
            "selective_risk",
            "selective_risk_increase",
        ]
    ]
    .copy()
)

primary_random_results.to_csv(
    TABLES_DIR
    / "Table_primary_all_feature_random_noise.csv",
    index=False,
    encoding="utf-8-sig",
)

primary_scale_results.to_csv(
    TABLES_DIR
    / "Table_primary_coherent_scale_bias.csv",
    index=False,
    encoding="utf-8-sig",
)

# Umbral descriptivo preespecificado:
# pérdida de macro-F1 de al menos 0.02.
degradation_rows = []

macro_f1_drop_summary = (
    random_noise_summary.loc[
        random_noise_summary[
            "metric"
        ] == "macro_f1_drop"
    ]
    .sort_values(
        [
            "dataset_id",
            "scenario",
            "noise_level_fraction_robust_scale",
        ]
    )
)

for (
    dataset_id,
    scenario_name,
), subset in macro_f1_drop_summary.groupby(
    [
        "dataset_id",
        "scenario",
    ],
    sort=True,
):
    crossing = subset.loc[
        subset[
            "mean"
        ] >= 0.02
    ]

    degradation_rows.append(
        {
            "dataset_id": dataset_id,
            "scenario": scenario_name,
            "predefined_macro_f1_drop_threshold": 0.02,
            "first_noise_level_reaching_threshold": (
                float(
                    crossing[
                        "noise_level_fraction_robust_scale"
                    ].iloc[0]
                )
                if len(crossing)
                else np.nan
            ),
            "threshold_reached_within_tested_range": bool(
                len(crossing)
            ),
            "interpretation": (
                "Descriptive crossing of a prespecified "
                "practical degradation threshold; not model selection."
            ),
        }
    )

degradation_thresholds = pd.DataFrame(
    degradation_rows
)

degradation_thresholds.to_csv(
    RESULTS_DIR
    / "06_prespecified_degradation_thresholds.csv",
    index=False,
    encoding="utf-8-sig",
)

display(primary_scale_results)
display(degradation_thresholds)

In [ ]:
# ============================================================
# 13. Figuras
# ============================================================
for dataset_id in DATASETS:
    # Sesgo coherente de escala.
    scale_plot = (
        scale_bias_results.loc[
            scale_bias_results[
                "dataset_id"
            ] == dataset_id
        ]
        .sort_values(
            "scale_bias_percent"
        )
    )

    figure, axis = plt.subplots(
        figsize=(8, 5)
    )

    axis.plot(
        scale_plot[
            "scale_bias_percent"
        ],
        scale_plot[
            "macro_f1"
        ],
        marker="o",
        label="Macro-F1",
    )

    axis.set_xlabel(
        "Sesgo coherente de escala (%)"
    )
    axis.set_ylabel(
        "Macro-F1"
    )
    axis.set_title(
        f"Sensibilidad a escala — {dataset_id}"
    )
    axis.grid(True, alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR
        / f"coherent_scale_bias_macro_f1_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        FIGURES_DIR
        / f"coherent_scale_bias_macro_f1_{dataset_id}.pdf",
        bbox_inches="tight",
    )

    plt.show()

    # Ruido aleatorio: pérdida de macro-F1.
    noise_plot = (
        random_noise_summary.loc[
            (
                random_noise_summary[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                random_noise_summary[
                    "metric"
                ] == "macro_f1_drop"
            )
        ]
        .copy()
    )

    figure, axis = plt.subplots(
        figsize=(9, 6)
    )

    for scenario_name, subset in (
        noise_plot.groupby(
            "scenario",
            sort=True,
        )
    ):
        subset = subset.sort_values(
            "noise_level_percent_robust_scale"
        )

        axis.plot(
            subset[
                "noise_level_percent_robust_scale"
            ],
            subset[
                "mean"
            ],
            marker="o",
            label=scenario_name,
        )

        axis.fill_between(
            subset[
                "noise_level_percent_robust_scale"
            ],
            subset[
                "interval_low"
            ],
            subset[
                "interval_high"
            ],
            alpha=0.15,
        )

    axis.axhline(
        0.02,
        linestyle="--",
        linewidth=1,
        label="Pérdida práctica = 0.02",
    )

    axis.set_xlabel(
        "Ruido (% de escala robusta de entrenamiento)"
    )
    axis.set_ylabel(
        "Pérdida de macro-F1"
    )
    axis.set_title(
        f"Ruido aleatorio morfométrico — {dataset_id}"
    )
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR
        / f"random_noise_macro_f1_drop_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        FIGURES_DIR
        / f"random_noise_macro_f1_drop_{dataset_id}.pdf",
        bbox_inches="tight",
    )

    plt.show()

    # Cobertura y riesgo selectivo.
    selective_plot = (
        random_noise_summary.loc[
            (
                random_noise_summary[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                random_noise_summary[
                    "scenario"
                ] == "all_features"
            )
            & (
                random_noise_summary[
                    "metric"
                ].isin(
                    [
                        "selective_coverage",
                        "selective_risk",
                    ]
                )
            )
        ]
        .copy()
    )

    figure, axis = plt.subplots(
        figsize=(9, 6)
    )

    for metric_name, subset in (
        selective_plot.groupby(
            "metric",
            sort=True,
        )
    ):
        subset = subset.sort_values(
            "noise_level_percent_robust_scale"
        )

        axis.plot(
            subset[
                "noise_level_percent_robust_scale"
            ],
            subset[
                "mean"
            ],
            marker="o",
            label=metric_name,
        )

    axis.set_xlabel(
        "Ruido en todas las variables "
        "(% de escala robusta)"
    )
    axis.set_ylabel(
        "Proporción"
    )
    axis.set_title(
        f"Estabilidad de clasificación selectiva — {dataset_id}"
    )
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR
        / f"selective_stability_all_feature_noise_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        FIGURES_DIR
        / f"selective_stability_all_feature_noise_{dataset_id}.pdf",
        bbox_inches="tight",
    )

    plt.show()

In [ ]:
# ============================================================
# 14. Protocolo, validaciones y manifiesto
# ============================================================
protocol = {
    "protocol_name": (
        "Grain_AI_MDPI_measurement_uncertainty_sensitivity_v1"
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "random_seed": RANDOM_SEED,
    "n_monte_carlo": N_MONTE_CARLO,
    "perturbation_interval": (
        PERTURBATION_INTERVAL
    ),
    "primary_unit": (
        "duplicate_group_id"
    ),
    "scale_bias_levels": (
        SCALE_BIAS_LEVELS
    ),
    "random_noise_levels_fraction_training_robust_scale": (
        RANDOM_NOISE_LEVELS
    ),
    "feature_blocks": FEATURE_BLOCKS,
    "random_scenarios": RANDOM_SCENARIOS,
    "selective_configuration": (
        selective_configuration.to_dict(
            orient="records"
        )
    ),
    "scale_bias_definition": {
        "dry_bean": (
            "Features transformed according to "
            "predefined physical scale exponents."
        ),
        "iniap_rice": (
            "log_area shifted by 2*log(1+bias); "
            "invariant descriptors unchanged."
        ),
    },
    "random_noise_definition": (
        "Independent additive Gaussian noise with "
        "feature-specific SD equal to level times "
        "training IQR/1.349; physical bounds applied."
    ),
    "restrictions": [
        "No model retraining.",
        "No hyperparameter reselection.",
        "No temperature recalibration.",
        "No selective-threshold recalibration.",
        "No perturbation level selected from test.",
        "Monte Carlo intervals describe perturbation variability, not sampling uncertainty.",
        "No camera, illumination, segmentation, or external robustness claim.",
    ],
}

(
    PROTOCOL_DIR
    / "measurement_uncertainty_protocol_v1.json"
).write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

expected_random_rows = sum(
    len(RANDOM_SCENARIOS[dataset_id])
    * len(RANDOM_NOISE_LEVELS)
    * N_MONTE_CARLO
    for dataset_id in DATASETS
)

expected_scale_rows = (
    len(DATASETS)
    * len(SCALE_BIAS_LEVELS)
)

zero_scale_rows = (
    scale_bias_results.loc[
        np.isclose(
            scale_bias_results[
                "relative_scale_bias"
            ],
            0.0,
        )
    ]
)

validation_checks = {
    "notebook07_all_checks_passed": bool(
        notebook07_validation[
            "passed"
        ].astype(bool).all()
    ),
    "notebook09_all_checks_passed": bool(
        notebook09_validation[
            "passed"
        ].astype(bool).all()
    ),
    "exactly_two_frozen_champions": (
        len(training_champions)
        == len(DATASETS)
    ),
    "models_reproduce_notebook05": bool(
        model_fidelity[
            "passed"
        ].astype(bool).all()
    ),
    "fixed_selective_configuration_complete": (
        len(selective_configuration)
        == len(DATASETS)
    ),
    "all_scale_bias_scenarios_complete": (
        len(scale_bias_results)
        == expected_scale_rows
    ),
    "zero_scale_bias_reproduces_baseline": bool(
        np.allclose(
            zero_scale_rows[
                "macro_f1"
            ].sort_values().to_numpy(),
            baseline_results[
                "macro_f1"
            ].sort_values().to_numpy(),
            atol=1e-12,
            rtol=0.0,
        )
    ),
    "all_random_noise_scenarios_complete": (
        len(random_noise_raw)
        == expected_random_rows
    ),
    "all_random_replicates_finite": bool(
        np.isfinite(
            random_noise_raw[
                [
                    "accuracy",
                    "macro_f1",
                    "mcc",
                    "label_flip_rate",
                    "mean_absolute_probability_shift",
                    "selective_coverage",
                ]
            ].to_numpy(dtype=float)
        ).all()
    ),
    "feature_blocks_cover_model_inputs": all(
        set(
            feature
            for block_features
            in FEATURE_BLOCKS[
                dataset_id
            ].values()
            for feature in block_features
        )
        == set(
            bundles[
                dataset_id
            ][
                "feature_columns"
            ]
        )
        for dataset_id in DATASETS
    ),
    "no_model_retraining": True,
    "no_test_based_threshold_recalibration": True,
    "no_test_based_perturbation_selection": True,
}

final_validation = pd.DataFrame(
    [
        {
            "check": check,
            "passed": passed,
        }
        for check, passed
        in validation_checks.items()
    ]
)

final_validation.to_csv(
    RESULTS_DIR
    / "07_final_measurement_uncertainty_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation[
    "passed"
].astype(bool).all():
    display(final_validation)

    raise RuntimeError(
        "Falló al menos una validación final."
    )

def sha256_file(path: Path):
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),
                "size_bytes": (
                    path.stat().st_size
                ),
                "sha256": (
                    sha256_file(path)
                ),
            }
        )

integrity_manifest = pd.DataFrame(
    manifest_rows
)

integrity_manifest.to_csv(
    RESULTS_DIR
    / "08_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Validación final: CORRECTA")
display(final_validation)

print(
    "Archivos registrados:",
    len(integrity_manifest),
)

In [ ]:
# ============================================================
# 15. Exportación a Drive y ZIP
# ============================================================
DRIVE_NOTEBOOK10_ROOT = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK10_MEASUREMENT_UNCERTAINTY_OUTPUTS"
)

if DRIVE_NOTEBOOK10_ROOT.exists():
    shutil.rmtree(
        DRIVE_NOTEBOOK10_ROOT
    )

shutil.copytree(
    OUTPUT_ROOT,
    DRIVE_NOTEBOOK10_ROOT,
)

FINAL_ZIP_PATH = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK10_MEASUREMENT_UNCERTAINTY_OUTPUTS.zip"
)

if FINAL_ZIP_PATH.exists():
    FINAL_ZIP_PATH.unlink()

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(
        OUTPUT_ROOT.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),
            )

print(
    "Directorio exportado:",
    DRIVE_NOTEBOOK10_ROOT,
)

print(
    "ZIP final:",
    FINAL_ZIP_PATH,
)

print(
    "Tamaño ZIP (MB):",
    round(
        FINAL_ZIP_PATH.stat().st_size
        / 1e6,
        2,
    ),
)

# Interpretación editorial

- El sesgo coherente de escala prueba sensibilidad a una transformación físicamente estructurada, no a ruido independiente.
- El ruido aleatorio de descriptores es un análisis de estrés en el espacio tabular.
- Una pérdida pequeña bajo sesgo coherente y una pérdida mayor bajo ruido independiente indicaría que la consistencia interna entre descriptores es importante.
- Los bloques deben interpretarse junto con el Notebook 09, porque variables correlacionadas pueden sustituirse entre sí.
- La cobertura y el riesgo selectivos se calculan con el score y el umbral congelados del Notebook 07.
- Una caída de cobertura selectiva bajo perturbación no debe corregirse reajustando el umbral con la prueba.
- Estos resultados no demuestran robustez ante dominios externos ni sustituyen una validación con mediciones repetidas reales.